In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Add, Activation
from tensorflow.keras.regularizers import l2
from scipy.spatial import ConvexHull
from matplotlib.path import Path
from tensorflow.keras.layers import Dropout
from tensorflow.keras import regularizers

import os
import random
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.initializers import GlorotUniform
from sklearn.neighbors import LocalOutlierFactor
from tensorflow.keras.layers import Input, Dense, Activation, Add, BatchNormalization, LeakyReLU
from scipy.spatial.distance import mahalanobis
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from adjustText import adjust_text 

In [2]:
df= pd.read_csv("dataset.csv",index_col=0)
df = df[df['MOF_ID'] != 'hMOF-1002818']

In [3]:
metals=['chem_metal_Sc', 'chem_metal_Ti',
       'chem_metal_V', 'chem_metal_Cr', 'chem_metal_Mn', 'chem_metal_Fe',
       'chem_metal_Co', 'chem_metal_Ni', 'chem_metal_Cu', 'chem_metal_Zn',
       'chem_metal_Y', 'chem_metal_Zr', 'chem_metal_Nb', 'chem_metal_Mo',
       'chem_metal_Tc', 'chem_metal_Ru', 'chem_metal_Rh', 'chem_metal_Pd',
       'chem_metal_Ag', 'chem_metal_Cd', 'chem_metal_Hf', 'chem_metal_Ta',
       'chem_metal_W', 'chem_metal_Re', 'chem_metal_Os', 'chem_metal_Ir',
       'chem_metal_Pt', 'chem_metal_Au', 'chem_metal_Hg', 'chem_metal_Al',
       'chem_metal_Ga', 'chem_metal_In', 'chem_metal_Sn', 'chem_metal_Pb',
       'chem_metal_Bi', 'chem_metal_La', 'chem_metal_Ce', 'chem_metal_Pr',
       'chem_metal_Nd', 'chem_metal_Sm', 'chem_metal_Eu', 'chem_metal_Gd',
       'chem_metal_Tb', 'chem_metal_Dy', 'chem_metal_Ho', 'chem_metal_Er',
       'chem_metal_Tm', 'chem_metal_Yb', 'chem_metal_Lu',]
nonmetals=['chem_num_atoms',
 'chem_volume',
 'chem_density',
 'chem_avg_atomic_mass',
 'chem_avg_electronegativity',
 'chem_electronegativity_variance',
 'chem_metal_fraction',
 'chem_num_unique_elements',
 'chem_metal_atom_count',
 'chem_volume_per_atom',
 'geo_surface_area_m2g',
 'geo_surface_area_m2cm3',
 'geo_void_fraction',
 'geo_pld',
 'geo_lcd',
 'link_linker_atom_fraction',
 'link_linker_bond_length_mean',
 'link_linker_bond_length_std',
 'link_metal_coord_number_mean',
 'topo_avg_node_connectivity',
 'topo_avg_ring_size',
 'topo_coordination_number_mean',
 'topo_degree_assortativity',
 'topo_degree_centrality_mean',
 'topo_graph_density',
 'topo_graph_entropy',
 'topo_graph_transitivity',
 'topo_largest_cc_fraction',
 'topo_node_connectivity_std',
 'topo_num_connected_components',
 'topo_num_edges',
 'topo_num_nodes']
features=nonmetals+metals

In [4]:
df_cleaned = df.dropna()
df_cleaned = df_cleaned.reset_index()
print("NaNs in X_scaled:", df_cleaned.isnull().sum().sum())

NaNs in X_scaled: 0


In [5]:
output_dir='jaccard for chemical features/'

In [6]:
m=3.5

params = {
    # Font family
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],

    # Font sizes
    "axes.labelsize": 10*m,
    "font.size": 10*m,
    "legend.fontsize": 10*m,
    "xtick.labelsize": 9*m,
    "ytick.labelsize": 9*m,

    # Style for axis labels (xlabel, ylabel)
    'axes.labelweight': 'bold',
    'axes.labelcolor': 'black',

    # General styles for other elements
    'font.weight': 'bold',       # Makes title, etc., bold
    'xtick.color': 'black',      # Sets tick label color
    'ytick.color': 'black',
    'legend.labelcolor': 'black'
}

plt.rcParams.update(params)

In [ ]:
latent_dim=16
step_size=2
randomstate=0
for condition in ['no metals', 'with metals']:
    if condition=='no metals':
        X_train_val_full, X_test_full = train_test_split(df_cleaned, test_size=0.2, random_state=randomstate)
        X_train_full, X_val_full = train_test_split(X_train_val_full, test_size=0.25, random_state=42)
        
        scaler_X = StandardScaler()
        scaler_X.fit(X_train_full[nonmetals])
        
        X_train = scaler_X.transform(X_train_full[nonmetals])
        X_train = pd.DataFrame(X_train, index=X_train_full.index, columns=nonmetals)
        
        X_val = scaler_X.transform(X_val_full[nonmetals])
        X_val = pd.DataFrame(X_val, index=X_val_full.index, columns=nonmetals)
        
        X_test = scaler_X.transform(X_test_full[nonmetals])
        X_test = pd.DataFrame(X_test, index=X_test_full.index, columns=nonmetals)
        
        X_full_dataset=pd.concat([X_train, X_val,X_test], axis=0)



        
    else:
        X_train_val_full, X_test_full = train_test_split(df_cleaned, test_size=0.2, random_state=rs)
        X_train_full, X_val_full = train_test_split(X_train_val_full, test_size=0.25, random_state=42)
        
        scaler_X = StandardScaler()
        scaler_X.fit(X_train_full[nonmetals])
        
        X_train_nonmetal_np = scaler_X.transform(X_train_full[nonmetals])
        X_train_nonmetal = pd.DataFrame(X_train_nonmetal_np, index=X_train_full.index, columns=nonmetals)
        X_train = pd.concat([X_train_nonmetal, X_train_full[metals]], axis=1)
        
        X_val_nonmetal_np = scaler_X.transform(X_val_full[nonmetals])
        X_val_nonmetal = pd.DataFrame(X_val_nonmetal_np, index=X_val_full.index, columns=nonmetals)
        X_val = pd.concat([X_val_nonmetal, X_val_full[metals]], axis=1)
        
        X_test_nonmetal_np = scaler_X.transform(X_test_full[nonmetals])
        X_test_nonmetal = pd.DataFrame(X_test_nonmetal_np, index=X_test_full.index, columns=nonmetals)
        X_test = pd.concat([X_test_nonmetal, X_test_full[metals]], axis=1)
        
        X_full_dataset=pd.concat([X_train, X_val,X_test], axis=0)
        

    print(f"\n===== Training Model: Latent Dim = {latent_dim}, Step Size = {step_size} =====\n")
    input_dim=X_train.shape[1]
    encoder_neurons = list(range(input_dim - step_size, latent_dim, -step_size))
    decoder_neurons = encoder_neurons[::-1]
    
    input_layer = Input(shape=(input_dim,))
    x = input_layer
    
    for neurons in encoder_neurons:
        x = Dense(neurons)(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
    
    encoded = Dense(latent_dim, name='latent_space')(x)
    x = encoded
    
    for neurons in decoder_neurons:
        x = Dense(neurons)(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
    
    output_layer = Dense(input_dim, activation='linear')(x)
    autoencoder = Model(inputs=input_layer, outputs=output_layer)
    
    optimizer = Adam(learning_rate=0.001)
    autoencoder.compile(optimizer=optimizer, loss='mse')
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=0)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10, min_lr=1e-5, verbose=0)
    
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, X_train)).batch(3200).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, X_val)).batch(3200).prefetch(tf.data.AUTOTUNE)
    
    history = autoencoder.fit(
        train_ds,
        epochs=200,
        validation_data=val_ds,
        callbacks=[early_stopping, reduce_lr],
        verbose=0
    )
        
    reconstructions_train = autoencoder.predict(X_train.values)
    reconstructions_test = autoencoder.predict(X_test.values)
    reconstructions_full = autoencoder.predict(X_full_dataset.values)
    
    errors_train = np.mean(np.square(X_train.values - reconstructions_train), axis=1)
    np.save(f'jaccard for chemical features/errors_train_{condition}.npy',errors_train)

In [8]:
list_errors=[]
for x in ['jaccard for chemical features/errors_train_no metals.npy',
         'jaccard for chemical features/errors_train_with metals.npy']:
    list_errors.append(np.load(x))
        

In [9]:
K_values = [50, 100, 200]

# Check if we have exactly two lists
if len(list_errors) != 2:
    print(f"Error: list_errors contains {len(list_errors)} items. This script expects exactly 2.")
else:
    # Get the two error lists
    errors_a = np.asarray(list_errors[0])
    errors_b = np.asarray(list_errors[1])

    print("Calculating Jaccard Index for Top-K Anomalies")
    print("---------------------------------------------")

    for k in K_values:
        # 1. Find the indices of the top-K anomalies for each model
        # np.argsort() sorts ascending, so [-k:] gives the indices of the K largest values.
        top_k_indices_a = set(np.argsort(errors_a)[-k:])
        top_k_indices_b = set(np.argsort(errors_b)[-k:])
        
        # 2. Calculate intersection and union
        intersection_count = len(top_k_indices_a.intersection(top_k_indices_b))
        union_count = len(top_k_indices_a.union(top_k_indices_b))
        
        # 3. Calculate Jaccard Index: |A intersect B| / |A union B|
        if union_count == 0:
            jaccard_score = 1.0 if intersection_count == 0 else 0.0
        else:
            jaccard_score = intersection_count / union_count
            
        print(f"K = {k}:")
        print(f"  - Intersection (Overlap): {intersection_count}")
        print(f"  - Union: {union_count}")
        print(f"  - Jaccard Index: {jaccard_score:.4f}\n")

Calculating Jaccard Index for Top-K Anomalies
---------------------------------------------
K = 50:
  - Intersection (Overlap): 33
  - Union: 67
  - Jaccard Index: 0.4925

K = 100:
  - Intersection (Overlap): 62
  - Union: 138
  - Jaccard Index: 0.4493

K = 200:
  - Intersection (Overlap): 112
  - Union: 288
  - Jaccard Index: 0.3889



In [13]:
from scipy.stats import spearmanr
correlation, p_value = spearmanr(list_errors[0][:], list_errors[1])

print(f"Spearman Rank Correlation (ρ): {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

Spearman Rank Correlation (ρ): 0.5942
P-value: 0.0000


In [37]:
errors_A=list_errors[0]
errors_B=list_errors[1]
df_A=pd.DataFrame(errors_A,columns=['errors']).sort_values(by='errors',ascending=False).iloc[:200]
df_B=pd.DataFrame(errors_B,columns=['errors']).sort_values(by='errors',ascending=False).iloc[:1000]
inds_A=df_A.index
inds_B=df_B.index

contain=0
for x in inds_A:
    for y in inds_B:
        if x==y:
            contain=contain+1
print(contain,contain/200*100)

191 95.5
